In [63]:
import numpy as np
import pandas as pd

df = pd.read_csv('heart.csv')
print(df.head())

   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  


In [1]:

sex_map            = {'M': 1, 'F': 0}
ExerciseAngina_map = {'Y': 1, 'N': 0}
ST_slope_map       = {'Up': 1, 'Flat': 0, 'Down': -1}

df['Sex']            = df['Sex'].map(sex_map)
df['ExerciseAngina'] = df['ExerciseAngina'].map(ExerciseAngina_map)
df['ST_Slope']       = df['ST_Slope'].map(ST_slope_map)

df = pd.get_dummies(df, columns=['ChestPainType'], prefix='ChestPainType', dtype=int)
df = pd.get_dummies(df, columns=['RestingECG'],    prefix='RestingECG',    dtype=int) 

print('-------------------------------')
print(df.head())
print('-------------------------------')

X = df.drop('HeartDisease', axis=1).values
Y = df['HeartDisease'].values

np.random.seed(42)
indices = np.random.permutation(len(X))
X = X[indices]
Y = Y[indices]

split   = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

numeric_indices = [0, 2, 3, 5, 7]
for i in numeric_indices:
    mean = X_train[:, i].mean()
    std  = X_train[:, i].std()
    X_train[:, i] = (X_train[:, i] - mean) / std
    X_test[:, i]  = (X_test[:, i]  - mean) / std
    
    
W = np.zeros(X_train.shape[1])
b = 0
# print(X_train[:10])
print('-------------------------------')
print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)
print("Sample row:", X_train[0])


NameError: name 'df' is not defined

In [65]:
def predict(X, W, b):
    linear_model = np.dot(X, W) + b
    y_predicted = 1 / (1 + np.exp(-linear_model))
    return y_predicted

def compute_cost(X, Y, W, b):
    m = len(Y)
    y_predicted = predict(X, W, b)
    cost = (-1 / m) * np.sum(Y * np.log(y_predicted) + (1 - Y) * np.log(1 - y_predicted))
    return cost

def compute_gradient(X, Y, W, b):
    m = len(Y)
    y_predicted = predict(X, W, b)
    dw = 0
    db = 0
    for i in range(m):
        dw += (y_predicted[i] - Y[i]) * X[i]
        db += (y_predicted[i] - Y[i])
    dw /= m
    db /= m
    return dw, db

def gradient_descent(X, Y, W, b, learning_rate, num_iterations):
    for i in range(num_iterations):
        dw, db = compute_gradient(X, Y, W, b)
        W = W - learning_rate * dw
        b = b - learning_rate * db
        cost = compute_cost(X, Y, W, b)
        if (i + 1) % 100 == 0:
            print(f"Iteration {i+1}/{num_iterations}, Cost: {cost}")
    return W, b  

In [66]:
print(compute_cost(X_train, Y_train, W, b))
print('----------------------')
W,b = gradient_descent(X_train, Y_train, W, b, learning_rate=0.01, num_iterations=2000)

0.6931471805599453
----------------------
Iteration 100/2000, Cost: 0.5551785716423907
Iteration 200/2000, Cost: 0.4933966837767879
Iteration 300/2000, Cost: 0.4595956671761213
Iteration 400/2000, Cost: 0.43836287037788135
Iteration 500/2000, Cost: 0.4237165132325407
Iteration 600/2000, Cost: 0.412923241734032
Iteration 700/2000, Cost: 0.40457563241009975
Iteration 800/2000, Cost: 0.39788108615526313
Iteration 900/2000, Cost: 0.39236094750533823
Iteration 1000/2000, Cost: 0.3877094153950011
Iteration 1100/2000, Cost: 0.38372182810674843
Iteration 1200/2000, Cost: 0.38025575940904055
Iteration 1300/2000, Cost: 0.3772087458124802
Iteration 1400/2000, Cost: 0.37450494106191845
Iteration 1500/2000, Cost: 0.372086800278068
Iteration 1600/2000, Cost: 0.36990971815993917
Iteration 1700/2000, Cost: 0.36793846622458454
Iteration 1800/2000, Cost: 0.3661447612523957
Iteration 1900/2000, Cost: 0.3645055655542836
Iteration 2000/2000, Cost: 0.3630018729380223


Train Accuracy: 85.15%
Test Accuracy: 84.24%


In [ ]:
# Training accuracy
y_train_pred_prob = predict(X_train, W, b)
y_train_pred = (y_train_pred_prob >= 0.5).astype(int)
train_accuracy = np.sum(y_train_pred == Y_train) / len(Y_train)
print(f"Train Accuracy: {train_accuracy * 100:.2f}%")

# Test accuracy
y_test_pred_prob = predict(X_test, W, b)
y_test_pred = (y_test_pred_prob >= 0.5).astype(int)
test_accuracy = np.sum(y_test_pred == Y_test) / len(Y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# --- Confusion matrix on TRAINING data ---
TP_train = np.sum((y_train_pred == 1) & (Y_train == 1))
TN_train = np.sum((y_train_pred == 0) & (Y_train == 0))
FP_train = np.sum((y_train_pred == 1) & (Y_train == 0))
FN_train = np.sum((y_train_pred == 0) & (Y_train == 1))

print("\n--- Train Confusion Matrix ---")
print(f"TP: {TP_train}, TN: {TN_train}, FP: {FP_train}, FN: {FN_train}")
assert TP_train + TN_train + FP_train + FN_train == len(Y_train)

# --- Confusion matrix on TEST data ---
TP_test = np.sum((y_test_pred == 1) & (Y_test == 1))
TN_test = np.sum((y_test_pred == 0) & (Y_test == 0))
FP_test = np.sum((y_test_pred == 1) & (Y_test == 0))
FN_test = np.sum((y_test_pred == 0) & (Y_test == 1))

print("\n--- Test Confusion Matrix ---")
print(f"TP: {TP_test}, TN: {TN_test}, FP: {FP_test}, FN: {FN_test}")
assert TP_test + TN_test + FP_test + FN_test == len(Y_test)

Train Accuracy: 85.15%
Test Accuracy: 84.24%

--- Train Confusion Matrix ---
TP: 370, TN: 255, FP: 63, FN: 46

--- Test Confusion Matrix ---
TP: 86, TN: 69, FP: 23, FN: 6
